<!-- NB_DOC_INTRO_V1 -->
# Bases de donnees vectorielles — FAISS, Chroma, Qdrant

> 📚 **Doc thematique** : [docs/07_BDD_DE.md](docs/07_BDD_DE.md) (Bases de donnees / Data Engineering / Web)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Les **vector databases** stockent et recuperent des **embeddings** (vecteurs denses) par similarite (cosinus, dot product, L2). Cle pour : RAG, recommandation, recherche semantique, deduplication.

Ce notebook execute **FAISS** (in-process, ultra-rapide) + presente Chroma, Qdrant en code de reference. Pour le pipeline RAG complet : [retrieval_BDD_Vectorielle.ipynb](./retrieval_BDD_Vectorielle.ipynb) et [NLP_LangChain_RAG.ipynb](./NLP_LangChain_RAG.ipynb).

## Plan

1. Setup + embeddings
2. FAISS basique (IndexFlatL2, IndexFlatIP)
3. FAISS HNSW (Hierarchical Navigable Small World, scale)
4. FAISS IVF (Inverted File, scale)
5. Chroma (in-process, langchain)
6. Qdrant (serveur Rust, filtres puissants)
7. Comparatif
8. Pieges + References


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Setup — embeddings jouet

In [ ]:
# Generer 10 000 embeddings synthetiques (dim 128) — simulent des document embeddings
N = 10_000
D = 128
embeddings = np.random.randn(N, D).astype("float32")
# Normaliser pour utiliser dot product = cosine
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
print(f"Shape : {embeddings.shape}, dtype : {embeddings.dtype}")

# Query = 1 embedding
query = embeddings[42:43].copy()   # on cherche le doc 42 → devrait etre top-1

## 2. FAISS — IndexFlatL2 (exact, brute force)

In [ ]:
try:
    import faiss
    print(f"FAISS : {faiss.__version__}")
    FAISS_OK = True
except ImportError:
    print("FAISS pas installe : uv add --group vector faiss-cpu")
    FAISS_OK = False

if FAISS_OK:
    import time

    # Index plat (exact, lent sur tres gros corpus mais OK ici)
    index = faiss.IndexFlatL2(D)
    index.add(embeddings)

    # Search top-5
    t0 = time.perf_counter()
    distances, indices = index.search(query, k=5)
    t = (time.perf_counter() - t0) * 1000

    print(f"Top 5 (IndexFlatL2) : {indices[0]} (search en {t:.2f} ms)")
    print(f"Distances : {distances[0].round(4)}")

## 3. FAISS — IndexFlatIP (cosine via dot product sur vecteurs normalises)

In [ ]:
if FAISS_OK:
    index_ip = faiss.IndexFlatIP(D)   # Inner Product
    index_ip.add(embeddings)
    similarities, ids = index_ip.search(query, k=5)
    print(f"Top 5 (cosine) : {ids[0]}, sims = {similarities[0].round(4)}")

## 4. FAISS — HNSW (approximatif, ultra-rapide scale)

**HNSW** (Hierarchical Navigable Small World) = graphe multi-niveaux pour ANN. Standard pour 100k-1B vecteurs.

In [ ]:
if FAISS_OK:
    M = 32   # nb de connexions par noeud (typique 16-64)
    index_hnsw = faiss.IndexHNSWFlat(D, M)
    index_hnsw.hnsw.efConstruction = 200    # qualite vs speed du build
    index_hnsw.hnsw.efSearch = 50            # qualite vs speed du query

    index_hnsw.add(embeddings)
    t0 = time.perf_counter()
    distances, indices = index_hnsw.search(query, k=5)
    t = (time.perf_counter() - t0) * 1000
    print(f"Top 5 (HNSW) : {indices[0]} (search en {t:.2f} ms)")

## 5. FAISS — IVF (Inverted File, partition de Voronoi)

In [ ]:
if FAISS_OK:
    # IVF : partitionne l'espace en n_clusters, ne cherche que dans les plus proches clusters
    n_clusters = 100
    quantizer = faiss.IndexFlatL2(D)
    index_ivf = faiss.IndexIVFFlat(quantizer, D, n_clusters, faiss.METRIC_L2)
    index_ivf.train(embeddings)
    index_ivf.add(embeddings)

    # nprobe = nb clusters a explorer (trade-off recall/speed)
    index_ivf.nprobe = 10
    distances, indices = index_ivf.search(query, k=5)
    print(f"Top 5 (IVF, nprobe=10) : {indices[0]}")

## 6. Chroma — in-process avec persistance

```python
# pip install chromadb
import chromadb

client = chromadb.Client()   # in-memory
# client = chromadb.PersistentClient(path="./chroma_db")   # disk persistant

collection = client.create_collection("my_docs")
collection.add(
    embeddings=embeddings.tolist(),
    documents=[f"doc {i}" for i in range(N)],
    ids=[str(i) for i in range(N)],
    metadatas=[{"source": "demo", "idx": i} for i in range(N)],
)

# Query
results = collection.query(
    query_embeddings=query.tolist(),
    n_results=5,
    where={"source": "demo"},      # filtre metadata
)
print(results)
```

Avantage Chroma : **API simple**, **filtres metadata** natifs, **persistance** disque, integration **LangChain**.


## 7. Qdrant — serveur Rust (production-grade)

```python
# pip install qdrant-client
# (Qdrant tourne en serveur, demarrer avec Docker : docker run -p 6333:6333 qdrant/qdrant)

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(url="http://localhost:6333")

# Creer collection
client.create_collection(
    collection_name="my_docs",
    vectors_config=VectorParams(size=128, distance=Distance.COSINE),
)

# Insert
client.upsert("my_docs", points=[
    PointStruct(id=i, vector=embeddings[i].tolist(),
                payload={"category": f"cat_{i % 4}"})
    for i in range(N)
])

# Search avec filtre
from qdrant_client.models import Filter, FieldCondition, MatchValue
results = client.search(
    collection_name="my_docs",
    query_vector=query[0].tolist(),
    query_filter=Filter(must=[FieldCondition(key="category", match=MatchValue(value="cat_2"))]),
    limit=5,
)
for hit in results:
    print(f"id={hit.id}, score={hit.score:.4f}")
```

Qdrant : **filtres puissants** (booleens complexes), **payload riche**, **REST + gRPC**, scale a milliards.


## 8. Comparatif

| Vector DB | Type | Pros | Cons | Quand |
|---|---|---|---|---|
| **FAISS** (Meta) | In-process C++ | Tres rapide, beaucoup d'algos | Pas de filtres metadata natifs | Recherche pure, fort throughput |
| **Chroma** | In-process / serveur | Pythonic, LangChain integration, persistance | Plus jeune, scale moyen | Dev rapide, POC |
| **Qdrant** | Serveur Rust | Filtres puissants, payload riche, REST+gRPC | Demande deploiement | Production avec metadata |
| **Pinecone** | SaaS | Zero-ops, scale | Vendor lock-in, cout | Si pas envie de gerer infra |
| **Milvus** | Serveur distribue | Scale milliards | Complexe a operer | Tres grand corpus |
| **Weaviate** | Serveur | Schema + hybrid search natif | Plus lourd | Quand tu veux GraphQL + vectors |
| **pgvector** | Postgres extension | Si deja Postgres en prod | Perf moindre que dedies | Eviter setup nouveau |

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| `IndexFlatL2` sur 10M+ vecteurs | Trop lent — HNSW ou IVF-PQ |
| Pas normaliser pour cosine | `embeddings /= np.linalg.norm(emb, axis=1, keepdims=True)` |
| Stocker float32 quand float16 OK | Gain 2× memoire |
| Recreer index a chaque restart | Persister (faiss.write_index) |
| Ignorer les filtres metadata | Souvent crucial (ex: filtrer par date, user) |
| Top-k = 100 sans re-ranking | Croiser avec cross-encoder pour qualite |
| Pas tester recall@k vs index exact | Mesurer la perte qualite d'ANN |


## References

### Documentation
- FAISS : https://github.com/facebookresearch/faiss
- Chroma : https://docs.trychroma.com/
- Qdrant : https://qdrant.tech/documentation/
- Pinecone : https://docs.pinecone.io/
- Milvus : https://milvus.io/docs

### Voir aussi
- [retrieval_BDD_Vectorielle.ipynb](retrieval_BDD_Vectorielle.ipynb)
- [NLP_LangChain_RAG.ipynb](NLP_LangChain_RAG.ipynb)
- [NLP_Recherche_d_informations.ipynb](NLP_Recherche_d_informations.ipynb)
